# Риск и распределение инвестиций в стартапы: классификация

---

# Цель:
- Написать модель машинного обучения для:
    - классификации стартапов (отказ/соглашение финансировать стартап);
    - выявление и интерпретация признаков, влияющих на отказ в финансировании;
    - сохранение модели для дальнейшего использования (расчёт объёма инвестиций на одобренные стсртапы).

In [1]:
import mlflow 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

C:\DataScienceProjects\startup_applications\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('../data/processed/startup_investment_dataset+rejected_status.csv')

df.head()

,startup_stage,industry,region,requested_amount,pre_money_valuation,team_size,founders_experience_years,annual_revenue,market_size_estimate,investment_amount,is_rejected
0,Pre-Seed,ClimateTech,US,1.381327e+06,7.223187e+06,9,3,0.000000,6.039942e+06,0.000000e+00,1
1,Series B,HealthTech,US,7.784759e+06,3.049307e+07,13,8,96087.461835,6.886030e+07,7.203092e+06,0
2,Pre-Seed,HealthTech,LATAM,1.306439e+06,1.022702e+07,5,3,0.000000,1.896208e+07,0.000000e+00,1
3,Idea,E-commerce,US,8.840830e+05,4.531614e+06,3,2,0.000000,7.717272e+06,0.000000e+00,1
4,Series B,E-commerce,US,1.608484e+07,5.189190e+07,10,7,133652.729965,1.415670e+07,1.672645e+07,0


In [3]:
df['is_rejected'].value_counts()

is_rejected
0    1948
1    1052
Name: count, dtype: int64

In [4]:
df = df.drop(columns=[
    'market_size_estimate', # TAM не используется для принятия решений об инвестировании;
    'investment_amount' # Объём инвестиций равен НУЛЮ, если в финансировании отказано
])

target = 'is_rejected' # Целевая переменная: соглашение - 0, отказ - 1

df.head(3)

,startup_stage,industry,region,requested_amount,pre_money_valuation,team_size,founders_experience_years,annual_revenue,is_rejected
0,Pre-Seed,ClimateTech,US,1.381327e+06,7.223187e+06,9,3,0.000000,1
1,Series B,HealthTech,US,7.784759e+06,3.049307e+07,13,8,96087.461835,0
2,Pre-Seed,HealthTech,LATAM,1.306439e+06,1.022702e+07,5,3,0.000000,1


In [5]:
print(f'Дупликатов: {df.duplicated().sum()}') # Дупликатов не обнаружено

Дупликатов: 0


In [6]:
# Модели для классификации
from sklearn.linear_model import LogisticRegression # Логистическая регрессия
from sklearn.ensemble import RandomForestClassifier # Случайный лес
from sklearn.svm import SVC # Классификатор (опорные векторы)

# Модели градиентного спуска
# from sklearn.ensemble import GradientBoostingClassifier # Классический градиентный бустинг (ОТМЕНЁН - низкая скорость обучения, слабая производительность)
from xgboost import XGBClassifier 
from lightgbm import LGBMClassifier

# Преобразование данных
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Пайплайн 
from sklearn.pipeline import Pipeline

# Разбиение данных (train/test)
from sklearn.model_selection import train_test_split, GridSearchCV

# Метрики оценки качества
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, f1_score 

# Подготовка данных / препроцессинг

In [9]:
def create_additional_features(df: pd.DataFrame):
    df = df.copy()
    
    eps = 1

    # Money features ratio

    df['requested_to_valuation'] = (
        df['requested_amount']
        / (df['pre_money_valuation'] + eps)
    )

    df['revenue_to_valuation'] = (
        df['annual_revenue']
        / (df['pre_money_valuation'] + eps)
    )

    df['revenue_per_employee'] = (
        df['annual_revenue']
        / (df['team_size'] + eps)
    )

    # Team features

    df['experience_per_member'] = (
        df['founders_experience_years']
        / (df['team_size'] + eps)
    )

    df['team_maturity'] = (
        df['team_size']
        * df['founders_experience_years']
    )

    # Business features

    df['early_stage'] = (
        df['startup_stage']
        .isin(['Idea', 'Pre-Seed'])
        .astype(int)
    )

    df['is_us_market'] = (
        (df['region'] == 'US')
        .astype(int)
    )

    # Logarifms 

    money_features = ['requested_amount', 'pre_money_valuation',
                      'annual_revenue', 'market_size_estimate']

    for col in money_features:
        if col in df.columns:
            df[f'{col}_log'] = np.log1p(df[col])

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    return df

In [10]:
df = create_additional_features(df)

In [11]:
X = df.drop(columns=['is_rejected'])
y = df['is_rejected']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=df['is_rejected'], test_size=.3, random_state=42)

In [12]:
num_features = X_train.select_dtypes(exclude=['object']).columns.tolist() # Числовые признаки
cat_features = X_train.select_dtypes(include=['object']).columns.tolist() # Категориальные признаки

print(f'Числовые признаки:\n{num_features}')
print('-' * 15)
print(f'Категориальные признаки:\n{cat_features}')

Числовые признаки:
['requested_amount', 'pre_money_valuation', 'team_size', 'founders_experience_years', 'annual_revenue', 'requested_to_valuation', 'revenue_to_valuation', 'revenue_per_employee', 'experience_per_member', 'team_maturity', 'early_stage', 'is_us_market', 'requested_amount_log', 'pre_money_valuation_log', 'annual_revenue_log']
---------------
Категориальные признаки:
['startup_stage', 'industry', 'region']


In [13]:
preprocessor = ColumnTransformer(transformers=[
    ('scaler', StandardScaler(), num_features),
    ('encoder', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'), cat_features)
])


preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scaler', ...), ('encoder', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``.

# Машинное обучение (модели)

### - Обучены модели машинного обучения, гиперпараметры оптимизированы по метрикам `roc-auc` и `f1-score`:

$$
F1 = 2 \cdot \frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}
$$ 

$$
\text{precision} = \frac{TP}{TP + FP}, \quad
\text{recall} = \frac{TP}{TP + FN}
$$

- \(TP\) — истинно положительные (правильно предсказанные «отказы»)  
- \(FP\) — ложноположительные (предсказали отказ, но его не было)  
- \(FN\) — ложноотрицательные (не предсказали отказ, а он был)

---

### ROC-AUC (Area Under ROC Curve)

ROC-AUC показывает, насколько хорошо модель различает классы независимо от выбранного порога классификации.

ROC-кривая строится на основе:

- **TPR (True Positive Rate, чувствительность / recall)**  
$$
TPR = \frac{TP}{TP + FN}
$$

- **FPR (False Positive Rate)**  
$$
FPR = \frac{FP}{FP + TN}
$$

ROC-AUC — это площадь под ROC-кривой:

$$
ROC\text{-}AUC = \int_0^1 TPR(FPR) \, d(FPR)
$$

In [14]:
import joblib
import os
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

interim_model_path = '../src/models/interim' 

--- 

- Модели будут логироваться с помощью MLFlow.
- Каждая модель и сетка гиперпараметров будут созданы отдельно в <a href='Блок инициализации'>Блоке инициализации</a>, обучаться и оцениваться в <a href='Блок обучения'>Блоке обучения</a>.
- Самая оптимальная модель сохранена в <a href='../src/models'>src/models</a>.

---

# Блок инициализации

In [15]:
def init_model(model_class, model_name='Model'):
    print(f'Модель {model_name} инициализирована.')
    return model_class

In [16]:
lr_model = init_model(
    LogisticRegression(max_iter=1000, solver='saga'),
    'LogisticRegression'
)

lr_grid = {
    'logreg__C': [0.01, 0.1, 0.5, 1],
    'logreg__penalty': ['l1', 'l2', 'elasticnet'],
    'logreg__l1_ratio': [0.2, 0.5, 0.8]
}

rf_model = init_model(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    'RandomForestClassifier'
)

rf_grid = {
    'rf__n_estimators': [200, 500],
    'rf__max_depth': [None, 10, 30],
    'rf__min_samples_split': [2, 10],
    'rf__min_samples_leaf': [1, 5],
    'rf__max_features': ['sqrt', 'log2'],
    'rf__criterion': ['gini', 'entropy'],
}

xgb_model = init_model(
    XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0),
    'XGBoostClassifier'
)

xgb_grid = {
    'xgb__n_estimators': [200, 400],
    'xgb__learning_rate': [0.01, 0.05, 0.1],
    'xgb__max_depth': [3, 5, 7],
    'xgb__min_child_weight': [1, 5, 10],
    'xgb__subsample': [0.7, 0.9],
    'xgb__colsample_bytree': [0.7, 0.9],
    'xgb__gamma': [0, 1, 5],
    'xgb__reg_alpha': [0, 0.1, 1],
    'xgb__reg_lambda': [1, 5],
}

lgb_model = init_model(
    LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
    'LightGBMClassifier'
)

lgbm_grid = {
    'lgbm__n_estimators': [200, 500],
    'lgbm__learning_rate': [0.05, 0.1],
    'lgbm__max_depth': [-1, 5, 10],
    'lgbm__num_leaves': [31, 63, 127],
    'lgbm__min_child_samples': [20, 50],
    'lgbm__subsample': [0.7, 0.9],
    'lgbm__colsample_bytree': [0.7, 0.9],
}

Модель LogisticRegression инициализирована.
Модель RandomForestClassifier инициализирована.
Модель XGBoostClassifier инициализирована.
Модель LightGBMClassifier инициализирована.


In [17]:
models_config = {
    "logreg": {
        "model": lr_model,
        "param_grid": lr_grid,
        "step_name": "logreg"
    },
    "rf": {
        "model": rf_model,
        "param_grid": rf_grid,
        "step_name": "rf"
    },
    "xgb": {
        "model": xgb_model,
        "param_grid": xgb_grid,
        "step_name": "xgb"
    },
    "lgbm": {
        "model": lgb_model,
        "param_grid": lgbm_grid,
        "step_name": "lgbm"
    }
}

# Блок обучения

In [18]:
from src.ml_utils import train_evaluate_model

In [ ]:
trained_models = {}
results = []

for model_name, config in models_config.items():

    result = train_evaluate_model(
        model_name=model_name,
        model=config["model"],
        param_grid=config["param_grid"],
        preprocessor=preprocessor,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        step_name=config["step_name"],
        scoring="roc_auc"
    )

    trained_models[model_name] = result["model"]

    results.append({
        "model": model_name,
        "cv_roc_auc": result["cv_score"],
        "test_f1": result["f1"]
    })

results_df = pd.DataFrame(results)


TRAINING: logreg
Fitting 5 folds for each of 36 candidates, totalling 180 fits


2026/05/20 01:05:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



TRAINING: rf
Fitting 5 folds for each of 96 candidates, totalling 480 fits


2026/05/20 01:07:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



TRAINING: xgb
Fitting 5 folds for each of 3888 candidates, totalling 19440 fits


2026/05/20 01:22:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



TRAINING: lgbm
Fitting 5 folds for each of 288 candidates, totalling 1440 fits


In [ ]:
results_df = results_df.sort_values(by="cv_roc_auc", ascending=False)

sns.barplot(data=results_df, x='model', y='cv_roc_auc')
plt.ylim(.65, .75)
plt.title('Результаты ROC-AUC на кросс-валидации')
plt.grid(axis='y')
plt.show()

display(results_df)

In [ ]:
best_model_name = results_df.iloc[0, 0]
best_model_roc_auc = results_df.iloc[0, 1]

print(f'Лучшая модель - {best_model_name}; ROC-AUC = {best_model_roc_auc}')

In [ ]:
model = trained_models[best_model_name]
display(model)

In [ ]:
model = trained_models[best_model_name]

y_score = model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.0, 1.01, 0.01)

best_threshold = 0
best_f1 = 0

for t in thresholds:
    y_pred = (y_score >= t).astype(int)

    f1 = f1_score(y_test, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print(f"Лучший threshold (вероятностный порог): {best_threshold:.2f}")
print(f"Лучший F1: {best_f1:.4f}")

# XGBoost Classifier - Best Overall 

- Threshold: **0.34**
- F1-Score: **0.63**
- ROC-AUC: **0.76**

## 1. Выбор модели

В рамках сравнения нескольких алгоритмов машинного обучения были протестированы:

- Logistic Regression  
- Random Forest  
- XGBoost  
- LightGBM  

### Результаты сравнения:

| Модель | CV ROC-AUC | Test F1 |
|--------|------------|---------|
| XGBoost | 0.7428 | 0.5694 |
| Logistic Regression | 0.7395 | 0.5829 |
| Random Forest | 0.7374 | 0.5779 |
| LightGBM | 0.7247 | 0.5439 |

### Вывод по выбору:

- XGBoost показал **лучший ROC-AUC на кросс-валидации**
- Logistic Regression показала **чуть лучший F1 на тесте**
- Однако XGBoost выбран как **best-overall model**, так как:
  - более устойчив на CV
  - лучше захватывает нелинейные зависимости
  - обеспечивает более гибкую настройку вероятностей (важно для threshold tuning)

---

## 2. Итоговая конфигурация модели (XGBoost)

```python
objective = 'binary:logistic'
colsample_bytree = 0.7
gamma = 1
learning_rate = 0.01
max_depth = 5
min_child_weight = 10
n_estimators = 400
reg_alpha = 1
reg_lambda = 5
subsample = 0.7
random_state = 42
n_jobs = -1
```

## 3. Расчёт вероятностного порога (threshold)

Был выполнен подбор порога классификации для улучшения баланса precision/recall.

**Самый лучший THRESHOLD = <span style='color: green'>0.34</span>**

---

# Интерпретация модели XGBoost

In [ ]:
xgb_model = model.named_steps["xgb"]

importances = xgb_model.feature_importances_

feature_names = model.named_steps["preprocessor"].get_feature_names_out()

fi_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(fi_df["feature"][:20][::-1], fi_df["importance"][:20][::-1])
plt.title("Самые влиятельные признаки (XGBoost)")
plt.show()

display(fi_df.head(20))

In [ ]:
import shap


xgb_model = model.named_steps["xgb"]
X_transformed = model.named_steps["preprocessor"].transform(X_test)

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_transformed)

In [ ]:
shap.summary_plot(
    shap_values,
    X_transformed,
    feature_names=model.named_steps["preprocessor"].get_feature_names_out()
)

## Интерпретация влияния признаков на решение о финансировании стартапа (XGBoost)

Анализ важности признаков модели XGBoost показал следующие ключевые закономерности в процессе принятия решений о финансировании стартапов.

---

### 1. Доминирующие количественные показатели стартапа

Наибольшее влияние на решение модели оказывают финансовые и командные метрики:

- `annual_revenue` (годовая выручка) — **наиболее сильный фактор модели**: высокий доход существенно снижает вероятность отказа в финансировании.
- `founders_experience_years` (опыт основателей) — опытная команда значительно повышает шансы на одобрение.
- `team_size` (размер команды) — более крупные команды воспринимаются как менее устойчивые и менее рискованные.
- `requested_amount` (запрашиваемый объём инвестиций) — согласно SHAP, увеличение суммы инвестиций повышает вероятность одобрения.
- `pre_money_valuation` (оценка стартапа до инвестиций) — заниженная оценка также может увеличивать инвестиционный риск.

---

### 2. Стадия развития стартапа

Стадия стартапа является важным структурным фактором риска:

- Ранние стадии (**Pre-Seed**) связаны с более высокой вероятностью отказа.
- Более зрелые стадии (**Seed**, **Series A**, **Series B**) снижают риск и повышают вероятность одобрения финансирования.

Это отражает естественную логику венчурного инвестирования, где зрелость бизнеса снижает неопределённость.

---

### 3. Отраслевые и географические факторы

Категориальные признаки оказывают умеренное, но стабильное влияние:

- Наибольший вклад среди отраслей наблюдается у **FinTech**, что указывает на повышенное внимание модели к данной категории.
- Другие отрасли (HealthTech, E-commerce, EdTech, ClimateTech) имеют сопоставимое, но менее выраженное влияние.
- Географические признаки показывают умеренное влияние:
  - стартапы из США (`region_US`) имеют несколько более благоприятный профиль,
  - Европа и LATAM демонстрируют относительно более высокий риск отказа.

---

### 4. Общая интерпретация модели

Модель демонстрирует, что принятие решений о финансировании в основном определяется следующими группами факторов:

- **Финансовая зрелость стартапа** (выручка, оценка, объём инвестиций)
- **Качество и опыт команды** (размер и опыт основателей)
- **Стадия развития проекта**
- **Дополнительные контекстные факторы** (индустрия и регион)

---

### Вывод

Модель XGBoost воспроизводит реалистичную логику венчурного инвестирования:

- ключевым фактором является **финансовая устойчивость стартапа**
- важную роль играет **компетентность команды**
- а также **стадия развития и контекст рынка**

Таким образом, модель можно интерпретировать как инструмент первичного риск-скоринга, который помогает инвестору быстро отсекать высокорисковые проекты на основе структурированных бизнес-признаков.

In [ ]:
try:
    joblib.dump(model, '../src/models/xgbclassifier_threshold=0_34.joblib')
    print('Финальная модель XGBoost сохранена в src/models/xgbclassifier_threshold=0_34.joblib.')
except Exception as e:
    print(f'Не удалось сохранить модель:\n{e}')